# 14 — Capstone：IT 支援工單處理 Agent

> 不需要 API key：跟前面大部分章節一樣，用 `scripted_model` 走離線劇本；唯一一個真的
> 對外互動的元件是 `12` 用過的本機 MCP server（stdio 子行程，不碰網路）。

## 今天要蓋出什麼

一句話：一個會分流、會查資料、遇到危險操作會先呈報主管的 IT 客服 agent。

把它想成一間 IT 客服中心：
- **前台主管（supervisor）**：先聽你描述問題，判斷是帳務問題還是技術問題，轉給對的專員
- **專員（billing / tech specialist）**：需要查資料就打電話到後台系統查（`ToolNode` /
  MCP 工具，對應 `05`、`12`）
- **遇到「動用比較大權限」的動作**（例如重啟服務）：專員不能自己決定，要先呈報給主管
  簽核（`interrupt()`，對應 `07`）
- **主管辦公室有一本工作日誌**：隨時記錄每張工單辦到哪一步（`checkpointer`，對應 `06`），
  暫停後也能從原地接著辦

## 這個場景動用了前面幾乎每一份 notebook 的技巧

| 用到的能力 | 對應章節 |
|---|---|
| `StateGraph` / `MessagesState` | `03` |
| `Command(goto=..., update=...)` 路由 | `04` |
| `ToolNode` + `tools_condition` + `handle_tool_errors=True` | `05` |
| `checkpointer` + `thread_id`（工單 = 一個 thread） | `06` |
| `interrupt()` + `Command(resume=...)`（核准重啟） | `07` |
| `stream_mode="updates"` | `08` |
| Supervisor pattern + subgraph（billing / tech 兩個專員） | `09` |
| 換 `SqliteSaver` 就能撐過程式重啟 | `10` |
| 掛 LangSmith tracing 不用改 graph 程式碼 | `11` |
| 工具來自本機 MCP server | `12` |

## 整體架構圖

這張圖跟下面實際組出來的 graph 結構完全對應——先看懂這張圖，後面每個 cell 都是在把圖裡
的某一塊組出來。

```
START
  │
  ▼
supervisor  ── Command(goto=...) 路由 (04) ──┐
                                              │
                       ┌──────────────────────┴───────────────────────┐
                       ▼                                              ▼
              billing_specialist                              tech_specialist  (subgraph, 09)
              billing_step (不用工具)                                  │
                       │                                              ▼
                       ▼                                        investigate ◀──┐
                      END                                             │        │
                                                          tools_condition (05)  │
                                                       ┌────────┴────────┐      │
                                                  還有 tool_calls    沒有了（path_map
                                                       │            改導向 human_approval）
                                                       ▼                 │      │
                                                     tools ──────────────┼──────┘
                                               (check_service_status /   │
                                                lookup_runbook，12 的     │
                                                MCP 工具)                 │
                                                                          ▼
                                                                  human_approval
                                                    ┌─────────────────────┴─────────────────────┐
                                              不是 RESTART_NEEDED                          RESTART_NEEDED
                                                    │                                             │
                                                    ▼                                    interrupt()（07）
                                                   END                          等 Command(resume="approve"/"reject")
                                                                                             │
                                                                              ┌──────────────┴──────────────┐
                                                                           approve                        reject
                                                                              │                              │
                                                                              ▼                              ▼
                                                                       apply_restart                    cancelled
                                                                              │                              │
                                                                              └──────────────┬───────────────┘
                                                                                            END
```

checkpointer 只掛在最外層的圖——為什麼不能讓 subgraph 自己也掛一個，後面組裝完
`tech_specialist` 之後會解釋。

## 場景設定
客服系統收到一張工單，丟給這個 agent：
1. **Supervisor** 判斷工單類型：帳務問題（billing）還是技術問題（technical）
2. **Billing specialist**：直接草擬一份說明回覆（不需要工具，不需要核准）
3. **Tech specialist**：呼叫內部工具調查（查服務狀態、查 runbook），如果診斷結果是
   「需要重啟服務」——這是有風險的操作——就**停下來等人工核准**，核准才真的執行

In [1]:
import sys
from typing import Literal

sys.path.insert(0, ".")
from _llm import get_llm, has_api_key, scripted_model

from langchain_core.messages import AIMessage, HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, interrupt

client = MultiServerMCPClient(
    {"demo": {"transport": "stdio", "command": sys.executable, "args": ["_mcp_server.py"]}}
)
mcp_tools = await client.get_tools()
tools_by_name = {t.name: t for t in mcp_tools}
tech_tools = [tools_by_name["check_service_status"], tools_by_name["lookup_runbook"]]
print("tech_specialist 可用的工具:", [t.name for t in tech_tools])

tech_specialist 可用的工具: ['check_service_status', 'lookup_runbook']


## 組裝：Supervisor + 兩個專職 subgraph

每個 build 函式都用「model 從外面傳進來」的寫法——這樣同一套架構，離線示範傳
`scripted_model(...)`，真的接模型就傳 `get_llm()`，graph 的結構完全不用改，跟 `02` 的
三層次對比是同一個道理。

In [2]:
def build_supervisor(router_model):
    def supervisor(state: MessagesState) -> Command[Literal["billing_specialist", "tech_specialist"]]:
        decision = router_model.invoke(state["messages"]).content.strip()
        target = "tech_specialist" if "tech" in decision else "billing_specialist"
        return Command(
            goto=target,
            update={"messages": [("ai", f"[supervisor] 分類為 {decision}，轉給 {target}")]},
        )

    return supervisor


def build_billing_specialist(model):
    def billing_step(state: MessagesState) -> dict:
        return {"messages": [model.invoke(state["messages"])]}

    builder = StateGraph(MessagesState)
    builder.add_node("billing_step", billing_step)
    builder.add_edge(START, "billing_step")
    builder.add_edge("billing_step", END)
    return builder.compile()

`tech_specialist` 內部是一個完整的 ReAct 迴圈（`05`：`investigate` ↔ `tools`）接上
human-in-the-loop（`07`）：調查 → 判斷要不要重啟 → 要的話中斷等人核准 → 核准才真的執行
「重啟」。

一個關鍵改動：`add_conditional_edges` 這裡自訂了 `path_map`——`tools_condition` 判斷
「沒有更多 tool_calls」時，預設會導去 `__end__`，這裡改成導去 `human_approval`。`04`
學過 `add_conditional_edges` 的 `path_map` 可以自訂目標，這正是它的實際用途：讓「工具都
呼叫完了」不等於「流程結束」，而是先繞去一個審核關卡。

`human_approval` 這個 node 裡暫停/等待核准的細節：

```
investigate 判斷完，最後一則訊息裡有沒有 "RESTART_NEEDED"？
        │
   ┌────┴────┐
   ▼         ▼
  沒有       有
   │         │
   ▼         ▼
  END   interrupt({"ask": "核准重啟服務嗎？", "context": ...})
              │
        graph 完全暫停在這裡，狀態被 checkpointer 存下來（06）
              │
        外部呼叫 agent.ainvoke(Command(resume="approve" 或 "reject"), config)
              │
         ┌────┴────┐
         ▼         ▼
      approve    reject
         │         │
         ▼         ▼
   apply_restart  cancelled
```

`interrupt()` 不是丟例外中斷程式——它是把整個 graph 執行「凍結」在這一步，靠 `checkpointer`
記住暫停時的狀態，等外面用同一個 `thread_id` 傳入 `Command(resume=...)` 才會從這裡接著跑
（`07` 學過的機制）。

In [3]:
def build_tech_specialist(investigate_model, tools):
    bound_model = investigate_model.bind_tools(tools)

    def investigate(state: MessagesState) -> dict:
        return {"messages": [bound_model.invoke(state["messages"])]}

    def human_approval(state: MessagesState) -> Command[Literal["apply_restart", "cancelled", "__end__"]]:
        last = state["messages"][-1].content
        if "RESTART_NEEDED" not in last:
            return Command(goto=END)  # 診斷結果不需要有風險的操作，直接結束
        decision = interrupt({"ask": "核准重啟服務嗎？", "context": last})
        return Command(goto="apply_restart" if decision == "approve" else "cancelled")

    def apply_restart(state: MessagesState) -> dict:
        return {"messages": [("ai", "服務已重啟並確認健康 (200 OK)")]}

    def cancelled(state: MessagesState) -> dict:
        return {"messages": [("ai", "人工駁回，未執行重啟，工單保留待人工處理")]}

    builder = StateGraph(MessagesState)
    builder.add_node("investigate", investigate)
    builder.add_node("tools", ToolNode(tools, handle_tool_errors=True))  # 05：錯誤處理陷阱，這裡明確開啟
    builder.add_node("human_approval", human_approval)
    builder.add_node("apply_restart", apply_restart)
    builder.add_node("cancelled", cancelled)
    builder.add_edge(START, "investigate")
    builder.add_conditional_edges("investigate", tools_condition, {"tools": "tools", "__end__": "human_approval"})
    builder.add_edge("tools", "investigate")
    builder.add_edge("apply_restart", END)
    builder.add_edge("cancelled", END)
    return builder.compile()


def build_ticket_agent(router_model, billing_model, investigate_model, tools, checkpointer):
    top = StateGraph(MessagesState)
    top.add_node("supervisor", build_supervisor(router_model))
    top.add_node("billing_specialist", build_billing_specialist(billing_model))
    top.add_node("tech_specialist", build_tech_specialist(investigate_model, tools))
    top.add_edge(START, "supervisor")
    top.add_edge("billing_specialist", END)
    top.add_edge("tech_specialist", END)
    # checkpointer 只掛在最外層——subgraph 內的 interrupt() 一樣能正常運作（06、07 都驗證過這個機制）
    return top.compile(checkpointer=checkpointer)


shared_checkpointer = InMemorySaver()

## 一個容易踩的坑：checkpointer 只能掛在最外層

`build_ticket_agent` 只有最外層的 `top.compile(checkpointer=checkpointer)` 掛了
checkpointer，`build_billing_specialist`、`build_tech_specialist` 這些 subgraph 自己
`compile()` 的時候都沒有掛。這不是漏掉，是故意的，也是必須的。

比喻：整棟大樓（最外層的 graph）只能有一本總樓管日誌（checkpointer），記錄每一層樓、
每個部門（每個 subgraph）現在辦到哪一步。如果每個部門自己又開一本日誌，暫停/恢復的
時候，樓管室（最外層的圖）根本不知道要去哪本日誌找正確的紀錄——`interrupt()` 需要靠
外層唯一的 checkpointer 才能對上「暫停在哪一步」，subgraph 自己再掛一個會讓這個對應
關係壞掉。

規則很簡單：**只在最外層 `compile(checkpointer=...)`，nested subgraph 一律不掛**，
`interrupt()` 才能正常暫停/恢復（`06`、`07` 都驗證過這個機制，這裡是它在多層 subgraph
下的實際應用）。

## 情境一：帳務工單——沒有風險操作，一次跑完

整個系列每個 tool_calls 劇本都只能用一次（`scripted_model` 的回應列表是一次性 iterator），
所以每個情境都建一個新的 agent 實例；`checkpointer` 是共用的，模擬同一個系統同時處理
多張工單。

In [4]:
billing_agent = build_ticket_agent(
    router_model=scripted_model(["billing"]),
    billing_model=scripted_model(["已產生帳單說明回覆，將於 24 小時內寄送給客戶。"]),
    investigate_model=scripted_model([]),  # billing 工單不會走到 tech_specialist
    tools=tech_tools,
    checkpointer=shared_checkpointer,
)

result = await billing_agent.ainvoke(
    {"messages": [HumanMessage("我上個月的帳單金額不對")]},
    {"configurable": {"thread_id": "TICKET-1001"}},
)
for m in result["messages"]:
    print(f"{type(m).__name__:12} | {m.content}")

HumanMessage | 我上個月的帳單金額不對
AIMessage    | [supervisor] 分類為 billing，轉給 billing_specialist
AIMessage    | 已產生帳單說明回覆，將於 24 小時內寄送給客戶。


## 情境二：技術工單——調查後判定需要重啟，人工核准

劇本：`investigate` 先呼叫 `check_service_status`（回報 DEGRADED），再呼叫 `lookup_runbook`
（建議重啟），最後輸出 `RESTART_NEEDED` 訊息觸發 `human_approval` 裡的 `interrupt()`。

In [5]:
def make_technical_scenario_agent():
    return build_ticket_agent(
        router_model=scripted_model(["technical"]),
        billing_model=scripted_model([]),
        investigate_model=scripted_model(
            [
                AIMessage(content="", tool_calls=[{"name": "check_service_status", "args": {"service": "billing-api"}, "id": "c1"}]),
                AIMessage(content="", tool_calls=[{"name": "lookup_runbook", "args": {"topic": "billing-api-latency"}, "id": "c2"}]),
                AIMessage(content="RESTART_NEEDED: billing-api - runbook 建議重啟。"),
            ]
        ),
        tools=tech_tools,
        checkpointer=shared_checkpointer,
    )


tech_agent_approve = make_technical_scenario_agent()
config_approve = {"configurable": {"thread_id": "TICKET-1002"}}

state = await tech_agent_approve.ainvoke(
    {"messages": [HumanMessage("billing-api 回應很慢，客戶在抱怨")]}, config_approve
)
print("暫停，等待核准:", state["__interrupt__"][0].value)

暫停，等待核准: {'ask': '核准重啟服務嗎？', 'context': 'RESTART_NEEDED: billing-api - runbook 建議重啟。'}


In [6]:
final_state = await tech_agent_approve.ainvoke(Command(resume="approve"), config_approve)
for m in final_state["messages"]:
    print(f"{type(m).__name__:12} | {m.content}")

HumanMessage | billing-api 回應很慢，客戶在抱怨
AIMessage    | [supervisor] 分類為 technical，轉給 tech_specialist
AIMessage    | 
ToolMessage  | [{'type': 'text', 'text': 'billing-api: DEGRADED - high latency on /invoices, restart recommended', 'id': 'lc_5af44b76-5838-4df1-b131-4822f8ca9628'}]
AIMessage    | 
ToolMessage  | [{'type': 'text', 'text': 'Runbook: restart billing-api pod, then verify /health returns 200.', 'id': 'lc_4819393c-3f10-4874-808e-e9d0d1975b3c'}]
AIMessage    | RESTART_NEEDED: billing-api - runbook 建議重啟。
AIMessage    | 服務已重啟並確認健康 (200 OK)


## 情境三：同樣的診斷，但人工駁回

換一張新工單（新 `thread_id`），劇本一樣走到 `RESTART_NEEDED`，這次核准動作換成 `"reject"`——
`human_approval` 裡的 `Command(goto="cancelled")` 分支被走到，不會真的執行重啟。

In [7]:
tech_agent_reject = make_technical_scenario_agent()
config_reject = {"configurable": {"thread_id": "TICKET-1003"}}

await tech_agent_reject.ainvoke({"messages": [HumanMessage("billing-api 回應很慢")]}, config_reject)
rejected_state = await tech_agent_reject.ainvoke(Command(resume="reject"), config_reject)
for m in rejected_state["messages"]:
    print(f"{type(m).__name__:12} | {m.content}")

HumanMessage | billing-api 回應很慢
AIMessage    | [supervisor] 分類為 technical，轉給 tech_specialist
AIMessage    | 
ToolMessage  | [{'type': 'text', 'text': 'billing-api: DEGRADED - high latency on /invoices, restart recommended', 'id': 'lc_ad3be969-1549-42e4-9a99-b48c51d78d49'}]
AIMessage    | 
ToolMessage  | [{'type': 'text', 'text': 'Runbook: restart billing-api pod, then verify /health returns 200.', 'id': 'lc_034dcd9d-518d-48e0-aa9f-14c7395c65f1'}]
AIMessage    | RESTART_NEEDED: billing-api - runbook 建議重啟。
AIMessage    | 人工駁回，未執行重啟，工單保留待人工處理


## 即時追蹤處理進度：`stream_mode="updates"`

正式系統裡，客服人員應該不是等整個流程跑完才看到結果，而是即時看到「supervisor 分類完了」
「specialist 在調查」「現在卡在等核准」。`08` 學過的 `stream_mode="updates"` 剛好給這種
輕量、逐步的進度更新。

In [8]:
stream_agent = make_technical_scenario_agent()
config_stream = {"configurable": {"thread_id": "TICKET-1004"}}

async for chunk in stream_agent.astream(
    {"messages": [HumanMessage("billing-api 很慢")]}, config_stream, stream_mode="updates"
):
    print(chunk)

{'supervisor': {'messages': [('ai', '[supervisor] 分類為 technical，轉給 tech_specialist')]}}


{'__interrupt__': (Interrupt(value={'ask': '核准重啟服務嗎？', 'context': 'RESTART_NEEDED: billing-api - runbook 建議重啟。'}, id='fd11814190932f0098955243911892ce'),)}


最後一個 chunk 是 `__interrupt__`——串流到這裡就停了，跟 `.ainvoke()` 的行為一致，只是
沿路能看到每一步的進度，而不是等到中斷那一刻才看到任何東西。

## 正式環境的下一步
這個 notebook 用的 `InMemorySaver`、`scripted_model`、本機 MCP server，換成正式環境的
版本時，**graph 的程式碼一行都不用改**：

- `InMemorySaver()` → `SqliteSaver` / `PostgresSaver`（`10`）：程式重啟、工單狀態還在
- `scripted_model(...)` → `get_llm()`（下一個 cell 示範）：真的模型自己判斷分類、自己
  決定要不要查 runbook
- 本機 stdio MCP server → 正式的 MCP server（可能是別台機器上的內部系統）：只要換
  `MultiServerMCPClient` 的連線設定（`transport`/`command`/`args` 換成 `url`）
- 環境變數設定 `LANGSMITH_TRACING=true`（`11`）：整條 supervisor → specialist → tool →
  interrupt 的流程會自動出現在同一棵 trace tree 裡，不用加任何程式碼

## 如果你有 API key：换成真的模型
架構完全一樣，只是把 `scripted_model(...)` 换成 `get_llm()`；supervisor 需要一個簡單的
系統提示詞才能穩定輸出分類結果。

In [9]:
if has_api_key():
    from langchain_core.messages import SystemMessage

    class RoutingModel:
        """Wrap get_llm() with a system prompt that forces a one-word category output."""

        def __init__(self, llm):
            self._llm = llm

        def invoke(self, messages):
            prompt = [SystemMessage("只回答 'billing' 或 'technical' 其中一個字，不要多餘文字。")] + list(messages)
            return self._llm.invoke(prompt)

    real_agent = build_ticket_agent(
        router_model=RoutingModel(get_llm()),
        billing_model=get_llm(),
        investigate_model=get_llm(),
        tools=tech_tools,
        checkpointer=InMemorySaver(),
    )
    real_result = await real_agent.ainvoke(
        {"messages": [HumanMessage("billing-api 回應很慢，客戶在抱怨")]},
        {"configurable": {"thread_id": "TICKET-REAL-1"}},
    )
    for m in real_result["messages"]:
        print(f"{type(m).__name__:12} | {m.content}")
    if "__interrupt__" in real_result:
        print("（真模型也判斷需要人工核准，流程在這裡停下來，跟劇本版本行為一致）")
else:
    print("尚未設定 OPENAI_API_KEY，跳過真模型呼叫（上面三個情境已經展示了完整流程）。")

尚未設定 OPENAI_API_KEY，跳過真模型呼叫（上面三個情境已經展示了完整流程）。


## 課程總結

這份 capstone 把 14 份 notebook 的重點收攏成一個可信的實際應用：一個會分流、會用工具
調查、對有風險的操作會停下來等人、處理過程可被持久化與追蹤的 agent。從 `00` 的環境設定、
`01` 的 LCEL，一路到這裡的 supervisor + subgraph + interrupt + MCP，你應該已經有能力：

- 判斷什麼時候該用 `create_agent` 一行打包、什麼時候需要手刻 `StateGraph`
- 設計一個 State schema、決定哪些欄位要用 reducer
- 幫工具接上 MCP，而不是把邏輯寫死在單一框架裡
- 在對的節點插入 human-in-the-loop，而不是讓 agent 自己執行所有操作
- 知道正式環境要換哪些元件（checkpointer、tracing），graph 邏輯本身不用重寫

剩下的就是拿你自己工作上真正的場景，把這個範本套上去。